## Divisão da planilha 'dataset_long.csv' em 7 novas planilhas

In [37]:
import pandas as pd

from pathlib import Path

# ---- Configuração ----
INPUT_FILE = Path('../data/dataset_long.csv')
# Os sete CSVs gerados ficam diretamente em data/, na raiz do projeto.
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colunas removidas de todos os arquivos de saída, além de 'key_area'
# (decisão do grupo — ver Card H)
COLUMNS_TO_DROP = ['run_id', 'tc_number', 'reason', 'active_limit', 'key_area']

# Componente excluído do Mainback: identificado como erro de sensor
# (ver discussão de outliers / seção 4.2.2 da documentação)
MAINBACK_EXCLUDE_TC = 'Oven RTD Connector Body'

In [38]:
def split_by_key_area(df: pd.DataFrame, output_dir: Path) -> tuple[dict, dict]:

    summary = {}
    area_dataframes = {}
    key_areas = df['key_area'].unique()
    print(f'Áreas encontradas ({len(key_areas)}): {list(key_areas)}\n')

    for area in key_areas:
        filename = output_dir / f"{area.lower().replace(' ', '_')}_data.csv"
        area_df = df[df['key_area'] == area].copy()

        if area == 'Mainback' and 'tc_name' in area_df.columns:
            area_df = area_df[area_df['tc_name'] != MAINBACK_EXCLUDE_TC]

        area_df = area_df.drop(columns=[c for c in COLUMNS_TO_DROP if c in area_df.columns])

        if 'temp' in area_df.columns:
            ordered_cols = [c for c in area_df.columns if c != 'temp'] + ['temp']
            area_df = area_df[ordered_cols]

        area_df.to_csv(filename, index=False)
        area_dataframes[area] = area_df
        summary[area] = len(area_df)
        print(f'"{filename.name}" salvo com {len(area_df)} linhas.')

    return summary, area_dataframes

In [39]:
df = pd.read_csv(INPUT_FILE)
print(f"Dataset principal carregado: {len(df)} linhas, {df['key_area'].nunique()} key_areas.\n")

summary, key_area_dataframes = split_by_key_area(df, OUTPUT_DIR)

# DataFrames finais, um para cada key_area.
mainback_df = key_area_dataframes['Mainback']
coil_temps_df = key_area_dataframes['Coil Temps']
main_vent_sys_df = key_area_dataframes['Main Vent Sys']
ipc_cooling_df = key_area_dataframes['IPC Cooling']
ipc_comp_df = key_area_dataframes['IPC Comp']
oven_hmi_df = key_area_dataframes['Oven HMI']
ct_knobs_df = key_area_dataframes['CT Knobs']

Dataset principal carregado: 4065 linhas, 7 key_areas.

Áreas encontradas (7): ['Mainback', 'Coil Temps', 'Main Vent Sys', 'IPC Cooling', 'IPC Comp', 'Oven HMI', 'CT Knobs']

"mainback_data.csv" salvo com 557 linhas.
"coil_temps_data.csv" salvo com 420 linhas.
"main_vent_sys_data.csv" salvo com 288 linhas.
"ipc_cooling_data.csv" salvo com 218 linhas.
"ipc_comp_data.csv" salvo com 1979 linhas.
"oven_hmi_data.csv" salvo com 220 linhas.
"ct_knobs_data.csv" salvo com 330 linhas.


In [40]:
# ---- Verificação de integridade ----
total_saved = sum(summary.values())
excluded_rows = len(df[df['tc_name'] == MAINBACK_EXCLUDE_TC]) if 'tc_name' in df.columns else 0
expected = len(df) - excluded_rows

print('Resumo por área:')
for area, n in summary.items():
    print(f'  {area}: {n} linhas')

print(f"\nTotal salvo: {total_saved} | Esperado (total original - excluídas): {expected}")
assert total_saved == expected, 'Divergência na contagem de linhas! Verifique o processamento.'
print('Contagem de linhas conferida com sucesso — nenhuma linha perdida ou duplicada além do esperado.')

Resumo por área:
  Mainback: 557 linhas
  Coil Temps: 420 linhas
  Main Vent Sys: 288 linhas
  IPC Cooling: 218 linhas
  IPC Comp: 1979 linhas
  Oven HMI: 220 linhas
  CT Knobs: 330 linhas

Total salvo: 4012 | Esperado (total original - excluídas): 4012
Contagem de linhas conferida com sucesso — nenhuma linha perdida ou duplicada além do esperado.


In [41]:
df.head(100)

,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
0,Run 1,1,"Door Lock Motor Ambient 1/2"" Below",Mainback,155,99.35,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
1,Run 1,2,Door Lock Microswitch Body,Mainback,136,114.72,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
2,Run 1,3,Convect Motor Winding,Mainback,165,89.93,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
3,Run 1,7,Storage Drawer (Metal Above),Mainback,150,167.87,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
4,Run 1,8,"L2, Wire, Power Cord",Mainback,60,53.65,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Run 2 - Pyro Flipped,40,IGBT 5,IPC Comp,110,91.99,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium
96,Run 2 - Pyro Flipped,41,IGBT 6,IPC Comp,110,91.96,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium
97,Run 2 - Pyro Flipped,42,IGBT 7,IPC Comp,110,91.96,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium
98,Run 2 - Pyro Flipped,43,IGBT 8,IPC Comp,110,91.99,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium


## Converter 'tc_name' em colunas

In [42]:
def tc_names_to_columns(area_df: pd.DataFrame) -> pd.DataFrame:
    """Transforma tc_name em colunas, preservando todas as temperaturas."""
    index_columns = [column for column in area_df.columns if column not in ['tc_name', 'temp']]
    pivot_source = area_df.copy()

    # Numera medições repetidas para não descartar temperaturas no pivot.
    pivot_source['measurement_number'] = (
        pivot_source.groupby(index_columns + ['tc_name'], dropna=False).cumcount()
    )

    wide_df = (
        pivot_source.pivot(
            index=index_columns + ['measurement_number'],
            columns='tc_name',
            values='temp',
        )
        .reset_index()
    )
    wide_df.columns.name = None

    # A numeração é apenas auxiliar para o pivot e não faz parte da planilha final.
    wide_df = wide_df.drop(columns='measurement_number')


    return wide_df

key_area_wide_dataframes = {
    area: tc_names_to_columns(area_df)
    for area, area_df in key_area_dataframes.items()
}

mainback_wide_df = key_area_wide_dataframes['Mainback']
coil_temps_wide_df = key_area_wide_dataframes['Coil Temps']
main_vent_sys_wide_df = key_area_wide_dataframes['Main Vent Sys']
ipc_cooling_wide_df = key_area_wide_dataframes['IPC Cooling']
ipc_comp_wide_df = key_area_wide_dataframes['IPC Comp']
oven_hmi_wide_df = key_area_wide_dataframes['Oven HMI']
ct_knobs_wide_df = key_area_wide_dataframes['CT Knobs']

for area, wide_df in key_area_wide_dataframes.items():
    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    wide_df.to_csv(filename, index=False)
    print(f'"{filename.name}" salvo com {len(wide_df)} linhas e {len(wide_df.columns)} colunas.')

"mainback_data.csv" salvo com 56 linhas e 17 colunas.
"coil_temps_data.csv" salvo com 56 linhas e 15 colunas.
"main_vent_sys_data.csv" salvo com 56 linhas e 14 colunas.
"ipc_cooling_data.csv" salvo com 55 linhas e 11 colunas.
"ipc_comp_data.csv" salvo com 56 linhas e 44 colunas.
"oven_hmi_data.csv" salvo com 55 linhas e 11 colunas.
"ct_knobs_data.csv" salvo com 55 linhas e 13 colunas.
